# Gaussian Shading 官方参考环境复现与受治理导入入口

该 Notebook 服务第二条链路: 官方参考环境复现 + 受治理导入。它不把基于 Stable Diffusion 2.1 的参考结果混入 SD3.5 主表, 只生成补充表所需的官方参考记录、环境报告、命令日志、schema、validation report 和 Google Drive 压缩包。

运行依赖: 不依赖主方法前序结果包, 但必须共享当前 `SLM_WM_PAPER_RUN_NAME` 派生的 prompt split、样本数和目标 FPR。四个 official reference 入口之间互不依赖, 可以在独立 Colab 会话中并行运行; 完整结果包闭合时会从 `external_baseline_official_reference` Drive 子目录读取这些包。

Notebook 通过 `paper_workflow/notebook_utils/notebook_entrypoint.py` 分派到 `paper_experiments/runners/gaussian_shading_official_reference.py`, 只负责 Colab 冷启动、参数注入、远程执行和打包。

## 运行边界

1. `probe_paper` 使用70个 Prompt 和 FPR=0.1, `pilot_paper` 使用700个 Prompt 和 FPR=0.01, `full_paper` 使用7000个 Prompt 和 FPR=0.001; 三个层级执行同一完整官方参考协议。
2. helper 按 `external_baseline/source_registry.json` 核验 Gaussian Shading 官方源码的精确 Git commit、远端地址和确定性补丁工作树。
3. 扩散模型固定为 `Manojb/stable-diffusion-2-1-base@0094d483a120f3f33dafbd187ea4aa60d10de75c`, 并在 `/content/slm_wm_model_sources` 共享快照中执行逐文件摘要核验。
4. Prompt 数据固定为 `Gustavosta/Stable-Diffusion-Prompts@d816d4a05cb89bde39dd99284c459801e1e7e69a`; 数据 revision 会进入源码补丁证据、正式记录和运行摘要。
5. OpenCLIP 固定使用 `ViT-g-14` 的本地 `open_clip_pytorch_model.bin`; runner 会核验来源 revision、文件大小、SHA-256 和共享快照摘要后再启动官方命令。
6. 官方脚本在固定依赖配置的隔离 Python 3.8 环境中运行, Notebook 只调用 repository helper, 不维护指标解析或证据生成逻辑。
7. 当前会话必须真实执行官方命令并返回0, 且检测、追踪、bit accuracy 与 CLIP 科学指标必须完整; 任一条件失败都会写出诊断并阻断正式记录与归档。


In [ ]:
SLM_WM_PAPER_RUN_NAME = "probe_paper"

import os

from google.colab import drive

drive.mount('/content/drive')
# 修改为 "full_paper" 可切换完整论文运行层级。
os.environ["SLM_WM_PAPER_RUN_NAME"] = SLM_WM_PAPER_RUN_NAME


In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

repository_url = os.environ.get("SLM_WM_REPOSITORY_URL", "https://github.com/RICHAAARC/SLM-WM.git")
repository_commit = os.environ.get("SLM_WM_REPOSITORY_COMMIT", "")
if re.fullmatch(r"[0-9a-f]{40}", repository_commit) is None:
    raise ValueError("正式运行前必须通过 SLM_WM_REPOSITORY_COMMIT 提供精确40位小写 Git SHA")
workspace_dir = Path(os.environ.get("SLM_WM_WORKSPACE_DIR", "/content/slm_wm_repository"))
workspace_dir.parent.mkdir(parents=True, exist_ok=True)
if workspace_dir.exists() and not (workspace_dir / ".git").is_dir():
    raise RuntimeError("SLM_WM_WORKSPACE_DIR 已存在但不是 Git 仓库")
if not (workspace_dir / ".git").is_dir():
    subprocess.run(["git", "clone", repository_url, str(workspace_dir)], check=True)

status_before_checkout = subprocess.run(
    ["git", "status", "--porcelain=v1", "--untracked-files=all"],
    cwd=workspace_dir,
    check=True,
    capture_output=True,
    text=True,
).stdout
if status_before_checkout:
    raise RuntimeError("复用 Colab 工作区前必须先清理 Git 工作树")
subprocess.run(["git", "fetch", "origin", "--tags", "--prune"], cwd=workspace_dir, check=True)
subprocess.run(["git", "rev-parse", "--verify", repository_commit + "^{commit}"], cwd=workspace_dir, check=True)
subprocess.run(["git", "checkout", "--detach", repository_commit], cwd=workspace_dir, check=True)

os.chdir(workspace_dir)
if str(workspace_dir) not in sys.path:
    sys.path.insert(0, str(workspace_dir))
from paper_workflow.colab_utils.formal_execution import verify_and_publish_formal_execution

formal_execution_lock = verify_and_publish_formal_execution(workspace_dir, repository_commit)
print({"workspace_dir": str(workspace_dir), "formal_execution_lock": formal_execution_lock})


In [ ]:
DEPENDENCY_PROFILE_ID = "workflow_orchestrator"
dependency_profile_result = subprocess.run(
    ["python", "scripts/prepare_dependency_profile.py", "--profile", DEPENDENCY_PROFILE_ID],
    check=True,
)
print({"dependency_profile_id": DEPENDENCY_PROFILE_ID, "return_code": dependency_profile_result.returncode})

from paper_workflow.colab_utils.dependency_check import build_notebook_dependency_report

dependency_report = build_notebook_dependency_report(DEPENDENCY_PROFILE_ID)
print(dependency_report)
assert dependency_report["dependency_decision"] == "pass", dependency_report


In [ ]:
from paper_workflow.colab_utils.paper_run_environment import configure_paper_run_environment
paper_run_environment = configure_paper_run_environment(
    workflow_name="official_reference_gaussian_shading",
)
print(paper_run_environment)


In [ ]:
import os

try:
    from google.colab import userdata
    token_from_secret = userdata.get('HF_TOKEN')
except Exception:
    token_from_secret = None

if token_from_secret and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = token_from_secret

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('huggingface_login_ready')
else:
    print('HF_TOKEN_not_found')


In [ ]:
import torch

device_report = {
    'cuda_available': torch.cuda.is_available(),
    'device_count': torch.cuda.device_count(),
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
}
print(device_report)


In [ ]:
import os
from pathlib import Path

from paper_workflow.notebook_utils.notebook_entrypoint import run_workflow

gaussian_shading_official_reference_summary = run_workflow(root='.', workflow_name="official_reference_gaussian_shading")
output_dir = Path('outputs/gaussian_shading_official_reference') / os.environ['SLM_WM_PAPER_RUN_NAME']
dependency_prepare_path = output_dir / 'gaussian_shading_dependency_environment_prepare_result.json'
model_prepare_path = output_dir / 'gaussian_shading_model_repository_prepare_result.json'
for diagnostic_path in (dependency_prepare_path, model_prepare_path):
    if diagnostic_path.exists():
        print(diagnostic_path)
        print(diagnostic_path.read_text(encoding='utf-8')[:6000])
gaussian_shading_official_reference_summary


In [ ]:
from paper_workflow.notebook_utils.notebook_entrypoint import package_workflow_outputs

archive_record = package_workflow_outputs(root='.', workflow_name="official_reference_gaussian_shading")
archive_record.to_dict()


In [ ]:
import os
from pathlib import Path

output_dir = Path('outputs/gaussian_shading_official_reference') / os.environ['SLM_WM_PAPER_RUN_NAME']

for result_path in sorted(output_dir.rglob('*')):
    if result_path.is_file() and result_path.suffix.lower() in {'.json', '.jsonl', '.txt'}:
        print(result_path)
        print(result_path.read_text(encoding='utf-8', errors='ignore')[:4000])

expected_sample_count = int(os.environ['SLM_WM_PAPER_RUN_EXPECTED_SAMPLE_COUNT'])
assert gaussian_shading_official_reference_summary['sample_count'] == expected_sample_count, gaussian_shading_official_reference_summary
if gaussian_shading_official_reference_summary['run_decision'] != 'pass':
    print({
        'gaussian_shading_official_reference_ready': gaussian_shading_official_reference_summary.get('gaussian_shading_official_reference_ready'),
        'unsupported_reason': gaussian_shading_official_reference_summary.get('unsupported_reason'),
        'dependency_environment_ready': gaussian_shading_official_reference_summary.get('dependency_environment_ready'),
        'official_command_return_code': gaussian_shading_official_reference_summary.get('official_command_return_code'),
    })
else:
    assert gaussian_shading_official_reference_summary['reference_import_ready'] is True, gaussian_shading_official_reference_summary
    assert gaussian_shading_official_reference_summary['governed_reference_record_count'] > 0, gaussian_shading_official_reference_summary
